# FlakeXplain v0.10: CodeBERT baseline, one outer fold per run

Protocol: `reports/llm-codebert-v0.10/PROTOCOL.md`. Run only after the protocol is frozen.

**Steps**
1. *Runtime → Change runtime type → T4 GPU*.
2. Set `FOLD` below (0, 1, 2, 3 or 4). Run **one fold per session**.
3. First time only: set `SMOKE = True` and run everything to check the setup (a few minutes). Its files end in `_smoke` and are not results.
4. Set `SMOKE = False` and *Runtime → Run all*. Upload `test_sources.jsonl.gz` when asked.
5. Download `fold_<k>_predictions.csv` and `fold_<k>_run.json` at the end, and put them in
   `local-evidence/llm-codebert-v0.10/` on the laptop.

The notebook never computes test-fold metrics. Scores are analysed locally after all five folds exist.

In [ ]:
FOLD = 0          # 0, 1, 2, 3 or 4
SMOKE = False     # True = quick setup check on a small subsample (not a result)

EXPECTED_SHA256 = "ff0e6d3b6fe8aaf98c74694a593453ff3fe58b4530e1d2302b3f3b4c3e823fd6"
DATA_FILE = "test_sources.jsonl.gz"
SEED = 20261021
MODEL_NAME = "microsoft/codebert-base"
MAX_LEN = 256
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
BATCH_SIZE = 32
VAL_FRACTION = 0.10
assert FOLD in (0, 1, 2, 3, 4)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
import hashlib, os
if not os.path.exists(DATA_FILE):
    from google.colab import files
    files.upload()
digest = hashlib.sha256(open(DATA_FILE, "rb").read()).hexdigest()
assert digest == EXPECTED_SHA256, f"Wrong input file: {digest}"
print("input file verified")

In [ ]:
import gzip, json, random, time
import numpy as np
from sklearn.model_selection import train_test_split

random.seed(SEED); np.random.seed(SEED)
rows = [json.loads(line) for line in gzip.open(DATA_FILE, "rt", encoding="utf-8")]
train_rows = [r for r in rows if r["outer_fold"] != FOLD]
test_rows = [r for r in rows if r["outer_fold"] == FOLD]
fit_rows, val_rows = train_test_split(train_rows, test_size=VAL_FRACTION, random_state=SEED,
                                      stratify=[r["label"] for r in train_rows])
if SMOKE:
    fit_rows = train_test_split(fit_rows, train_size=2000, random_state=SEED,
                                stratify=[r["label"] for r in fit_rows])[0]
    val_rows = train_test_split(val_rows, train_size=1000, random_state=SEED,
                                stratify=[r["label"] for r in val_rows])[0]
    test_rows = test_rows[:1000]
    EPOCHS = 1
test_projects = sorted({r["project"] for r in test_rows})
fit_pos = sum(r["label"] for r in fit_rows)
print(f"fold {FOLD}: fit {len(fit_rows)} ({fit_pos} flaky), val {len(val_rows)} "
      f"({sum(r['label'] for r in val_rows)} flaky), test {len(test_rows)} tests from {len(test_projects)} projects")
assert not {r["project"] for r in train_rows} & set(test_projects), "project leakage"

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(rs):
    enc = tokenizer([r["code"] for r in rs], truncation=True, max_length=MAX_LEN)
    return [(enc["input_ids"][i], r["label"]) for i, r in enumerate(rs)]

fit_data, val_data, test_data = encode(fit_rows), encode(val_rows), encode(test_rows)
truncated = sum(len(tokenizer(r["code"])["input_ids"]) > MAX_LEN for r in fit_rows[:5000])
print(f"tokenised; {truncated} of the first 5,000 training tests exceed {MAX_LEN} tokens")

In [ ]:
import math
from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score

pad_id = tokenizer.pad_token_id

def collate(batch):
    width = max(len(ids) for ids, _ in batch)
    ids = torch.full((len(batch), width), pad_id, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, (seq, _) in enumerate(batch):
        ids[i, :len(seq)] = torch.tensor(seq); mask[i, :len(seq)] = 1
    labels = torch.tensor([y for _, y in batch], dtype=torch.float)
    return ids, mask, labels

def predict(model, data):
    model.eval(); out = []
    with torch.no_grad():
        for ids, mask, _ in DataLoader(data, batch_size=128, collate_fn=collate):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=ids.cuda(), attention_mask=mask.cuda()).logits.squeeze(-1)
            out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).cuda()
neg = len(fit_data) - fit_pos
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / fit_pos, device="cuda"))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps = EPOCHS * math.ceil(len(fit_data) / BATCH_SIZE)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * steps), steps)
scaler = torch.cuda.amp.GradScaler()
loader = DataLoader(fit_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                    generator=torch.Generator().manual_seed(SEED))
val_labels = np.array([y for _, y in val_data])
best_ap, best_epoch, best_state, epoch_log = -1.0, None, None, []
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train(); total = 0.0
    for step, (ids, mask, labels) in enumerate(loader, 1):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=ids.cuda(), attention_mask=mask.cuda()).logits.squeeze(-1)
        loss = loss_fn(logits.float(), labels.cuda())
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item()
        if step % 500 == 0:
            print(f"epoch {epoch} step {step}/{len(loader)} loss {total / step:.4f} ({time.time() - start:.0f}s)")
    val_ap = float(average_precision_score(val_labels, predict(model, val_data)))
    epoch_log.append({"epoch": epoch, "train_loss": total / len(loader), "val_ap": val_ap,
                      "elapsed_s": round(time.time() - start)})
    print(epoch_log[-1])
    if val_ap > best_ap:
        best_ap, best_epoch = val_ap, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

In [ ]:
from sklearn.metrics import precision_recall_curve
import csv, platform, transformers, sklearn

model.load_state_dict(best_state)
val_scores = predict(model, val_data)
prec, rec, thr = precision_recall_curve(val_labels, val_scores)
f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
threshold = float(thr[int(np.argmax(f1))])
test_scores = predict(model, test_data)   # scores only; no test metrics here

suffix = "_smoke" if SMOKE else ""
pred_file = f"fold_{FOLD}_predictions{suffix}.csv"
with open(pred_file, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["example_id", "score"])
    for r, s in zip(test_rows, test_scores):
        w.writerow([r["example_id"], f"{s:.8f}"])
run = {"fold": FOLD, "smoke": SMOKE, "input_sha256": digest, "model": MODEL_NAME, "max_len": MAX_LEN,
       "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY, "warmup_fraction": WARMUP_FRACTION,
       "batch_size": BATCH_SIZE, "seed": SEED, "fit": len(fit_data), "fit_positives": fit_pos,
       "val": len(val_data), "val_positives": int(val_labels.sum()), "test": len(test_data),
       "test_projects": test_projects, "epoch_log": epoch_log, "best_epoch": best_epoch,
       "best_val_ap": best_ap, "threshold": threshold, "train_seconds": round(time.time() - start),
       "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
       "transformers": transformers.__version__, "sklearn": sklearn.__version__,
       "python": platform.python_version()}
run_file = f"fold_{FOLD}_run{suffix}.json"
json.dump(run, open(run_file, "w"), indent=1)
print(json.dumps({k: run[k] for k in ("fold", "best_epoch", "best_val_ap", "threshold", "train_seconds")}, indent=1))

In [ ]:
from google.colab import files
files.download(pred_file)
files.download(run_file)